# Tutorial 7 — Cohort Selection

This notebook derives **patient cohorts** from the feature table produced in Tutorial 6.
Each cohort is a filtered subset of patients meeting a specific demographic or clinical
criterion. The result is three registered datasets on the FCP, each ready for downstream
analytics or model training.

Cohort selection runs as **Code Objects** — the filtering logic executes on the
Rhino client. Raw patient records never reach this notebook.

**Can I do this in the UI instead?**
Yes — you can create and run each Code Object from the FCP Dashboard.
Navigate to Dashboard → Code → New Code Object → Python Code, paste the script,
assign the Patient Features input schema, and click Run.

---
**Prerequisites:**
- Tutorial 6 complete — `FEATURE_DATASET_UID` required in the Configuration cell
- Login information (username & password)

**Inputs:** Patient Features dataset (output of Tutorial 6)

**Outputs:** Three cohort datasets registered on the FCP. UIDs available in the Summary cell.

## Step 1: Configuration

In [ ]:
import json
import rhino_health as rh
from rhino_health.lib.endpoints.code_object.code_object_dataclass import (
    CodeObjectCreateInput,
    CodeObjectRunInput,
    CodeTypes,
)
from rhino_health.lib.endpoints.code_run.code_run_dataclass import CodeRunStatus
from rhino_health.lib.metrics import Count, Mean
from getpass import getpass
from rhino_health import ApiEnvironment

# From Tutorial 6 — paste your values here
PROJECT_UID          = "<YOUR_PROJECT_UID>"     # REPLACE
FEATURE_DATASET_UID  = "<FEATURE_DATASET_UID>"  # REPLACE
FEATURE_SCHEMA_UID   = "<FEATURE_SCHEMA_UID>"   # REPLACE

required = {
    "PROJECT_UID": PROJECT_UID,
    "FEATURE_DATASET_UID": FEATURE_DATASET_UID,
    "FEATURE_SCHEMA_UID": FEATURE_SCHEMA_UID,
}
for name, val in required.items():
    if val.startswith("<"):
        raise ValueError(f"Please fill in {name} before running this cell.")
    print(f"{name} = {val}")

## Step 2: Initialize Shared Utilities

Run this cell once. It defines helper functions used throughout this notebook.

> Be sure to replace `my_username` with your Rhino Health username!

In [ ]:
def authenticate():
    """Authenticate and return a session."""
    my_username = "<YOUR_USERNAME>" # REPLACE
    if my_username in ("", "<YOUR_EMAIL>", "<YOUR_USERNAME>"):
        raise ValueError("Please fill in your username in the authenticate() function.")
    session = rh.login(
        username=my_username,
        password=getpass(),
        rhino_api_url=ApiEnvironment.PROD_AWS_URL,
    )
    print(f"Logged in as <{my_username}>.")
    return session


def register_or_reuse_code_object(session, name, description, input_schema_uids, project_uid):
    """
    Register a Generalized Compute Code Object if one with this name doesn't already exist.

    Uses the generic-python-runner container. The actual Python script is passed at run time
    via run_params — not embedded here — so the same Code Object can be re-run with any script
    without re-registration.
    """
    existing = session.code_object.get_code_object_by_name(name, project_uid=project_uid)
    if existing:
        print(f"Code Object '{name}' already exists — reusing: {existing.uid}")
        return existing

    co = session.code_object.create_code_object(CodeObjectCreateInput(
        name=name,
        description=description,
        project_uid=project_uid,
        code_type=CodeTypes.GENERALIZED_COMPUTE,
        config={
            "container_image_uri": session.get_container_image_uri(
                "generic-python-runner", rhino_common_image=True
            ),
        },
        input_data_schema_uids=input_schema_uids,
        output_data_schema_uids=[None],  # auto-infer output schema
    ))
    print(f"Code Object registered: {co.uid}")
    return co


def run_and_poll(session, code_object_uid, input_dataset_uid, output_name,
                 run_params=None, max_wait=600):
    """
    Execute a Code Object with a single input dataset and wait for completion.

    run_params: dict passed to the container at runtime — use {"code": "..."} to supply
                the Python script for generic-python-runner.
    """
    async_response = session.code_object.run_code_object(CodeObjectRunInput(
        code_object_uid=code_object_uid,
        input_dataset_uids=[[input_dataset_uid]],
        output_dataset_naming_templates=[output_name],
        run_params=json.dumps(run_params) if run_params else None,
        timeout_seconds=max_wait,
    ))

    code_run_uid = async_response.code_run_uid
    print(f"Run initiated: {code_run_uid}")
    print("FCP UI: Dashboard → Code Runs → find this UID to monitor progress")

    code_run = session.code_run.get_code_run(code_run_uid)
    result = code_run.wait_for_completion(timeout_seconds=max_wait, print_progress=True)

    if result.status not in (CodeRunStatus.COMPLETED, CodeRunStatus.HALTED_SUCCESS):
        raise RuntimeError(f"Run ended with status: {result.status.value}")

    print("Run completed successfully.")
    return result


print("Utility functions loaded.")

In [ ]:
# Log in to Rhino Health and create a session object to interact with the API
session = authenticate()

---
## Step 3: Define the Cohort Filtering Scripts

Below, we define three Python scripts — one per cohort. Each script runs **inside the
`generic-python-runner` container on the Rhino client** and is passed at run time via
`run_params["code"]`. Scripts can be changed and re-run without re-registering the Code Object.

The Patient Features dataset is mounted at `/input/dataset.csv` (single-input). Each script
reads it, applies a filter, writes the cohort to `/output/dataset.csv`, and sets
`outputs = [[cohort]]` so the container can register the result.

| Cohort | Filter |
|---|---|
| Males 30–40 | `age >= 30 AND age <= 40 AND is_male == 1` |
| Females 30–40 | `age >= 30 AND age <= 40 AND is_female == 1` |
| Race 8515 | `race_concept_id == 8515` (OMOP: Asian) |

In [ ]:
from string import Template

_COHORT_TEMPLATE = Template("""\
import pandas as pd
import os

features_df = pd.read_csv("/input/dataset.csv")

n_input = len(features_df)
cohort = features_df[$filter_expr].copy()

print("Cohort: $label")
print("Input rows:  {}".format(n_input))
print("Output rows: {}".format(len(cohort)))
$extra_checks
os.makedirs("/output", exist_ok=True)
cohort.to_csv("/output/dataset.csv", index=False)
print("Output written: /output/dataset.csv")

outputs = [[cohort]]
""")


def make_cohort_code(label, filter_expr, extra_checks=""):
    """Build cohort filtering code to pass via run_params['code']."""
    return _COHORT_TEMPLATE.substitute(label=label, filter_expr=filter_expr, extra_checks=extra_checks)

In [ ]:
COHORT_MALES_30_40_CODE = make_cohort_code(
    label="Males aged 30-40",
    filter_expr='(features_df["age"] >= 30) & (features_df["age"] <= 40) & (features_df["is_male"] == 1)',
    extra_checks="""\
if len(cohort):
    print("Age range: {}-{}".format(cohort["age"].min(), cohort["age"].max()))
    print("is_male values: {}".format(cohort["is_male"].unique().tolist()))
else:
    print("WARNING: cohort is empty")""",
)
print("Cohort script defined: COHORT_MALES_30_40_CODE")

In [ ]:
COHORT_FEMALES_30_40_CODE = make_cohort_code(
    label="Females aged 30-40",
    filter_expr='(features_df["age"] >= 30) & (features_df["age"] <= 40) & (features_df["is_female"] == 1)',
    extra_checks="""\
if len(cohort):
    print("Age range: {}-{}".format(cohort["age"].min(), cohort["age"].max()))
    print("is_female values: {}".format(cohort["is_female"].unique().tolist()))
else:
    print("WARNING: cohort is empty")""",
)
print("Cohort script defined: COHORT_FEMALES_30_40_CODE")

In [ ]:
COHORT_RACE_8515_CODE = make_cohort_code(
    label="race_concept_id = 8515 (OMOP: Asian)",
    filter_expr='features_df["race_concept_id"] == 8515  # OMOP: Asian',
    extra_checks="""\
if not len(cohort):
    print("WARNING: cohort is empty — no patients with race_concept_id=8515.")
    print("  Check that Race was mapped to OMOP concept IDs in Tutorial 4 harmonization.")""",
)
print("Cohort script defined: COHORT_RACE_8515_CODE")

---
## Step 4: Register and Run

This step registers all three **Generalized Compute** Code Objects (once) and runs each one
against the Patient Features dataset. The `generic-python-runner` container is specified at
registration — each cohort script is passed separately as `run_params["code"]` at run time,
so scripts can be updated and re-run without re-registering.

> **If a Code Object already exists from a previous run**, it will be reused automatically.
> Re-running the register cell with a changed script has no effect on the stored definition —
> just re-run the relevant run cell to pick up the new code.

> **If you see a 503 build-unavailable error**, the platform's container build service is
> temporarily down. Wait a few minutes and try again, or contact support@rhinohealth.com.

In [ ]:
males_co = register_or_reuse_code_object(
    session,
    name="Cohort Selection — Males 30-40",
    description="Filter Patient Features to male patients aged 30-40.",
    input_schema_uids=[FEATURE_SCHEMA_UID],
    project_uid=PROJECT_UID,
)

print("--- Running: Males 30-40 ---")
males_run = run_and_poll(
    session,
    code_object_uid=males_co.uid,
    input_dataset_uid=FEATURE_DATASET_UID,
    output_name="Cohort — Males 30-40",
    run_params={"code": COHORT_MALES_30_40_CODE},
)
males_ds = session.dataset.get_dataset(males_run.output_dataset_uids.root[0].root[0].root[0])
COHORT_MALES_UID        = males_ds.uid
COHORT_MALES_SCHEMA_UID = males_ds.data_schema_uid
print(f"Males cohort dataset UID:  {COHORT_MALES_UID}")
print()

In [ ]:
females_co = register_or_reuse_code_object(
    session,
    name="Cohort Selection — Females 30-40",
    description="Filter Patient Features to female patients aged 30-40.",
    input_schema_uids=[FEATURE_SCHEMA_UID],
    project_uid=PROJECT_UID,
)

print("--- Running: Females 30-40 ---")
females_run = run_and_poll(
    session,
    code_object_uid=females_co.uid,
    input_dataset_uid=FEATURE_DATASET_UID,
    output_name="Cohort — Females 30-40",
    run_params={"code": COHORT_FEMALES_30_40_CODE},
)
females_ds = session.dataset.get_dataset(females_run.output_dataset_uids.root[0].root[0].root[0])
COHORT_FEMALES_UID        = females_ds.uid
COHORT_FEMALES_SCHEMA_UID = females_ds.data_schema_uid
print(f"Females cohort dataset UID: {COHORT_FEMALES_UID}")
print()

In [ ]:
race_co = register_or_reuse_code_object(
    session,
    name="Cohort Selection — Race 8515",
    description="Filter Patient Features to patients with race_concept_id = 8515 (OMOP: Asian).",
    input_schema_uids=[FEATURE_SCHEMA_UID],
    project_uid=PROJECT_UID,
)

print("--- Running: Race 8515 ---")
race_run = run_and_poll(
    session,
    code_object_uid=race_co.uid,
    input_dataset_uid=FEATURE_DATASET_UID,
    output_name="Cohort — Race 8515",
    run_params={"code": COHORT_RACE_8515_CODE},
)
race_ds = session.dataset.get_dataset(race_run.output_dataset_uids.root[0].root[0].root[0])
COHORT_RACE_UID        = race_ds.uid
COHORT_RACE_SCHEMA_UID = race_ds.data_schema_uid
print(f"Race 8515 cohort dataset UID: {COHORT_RACE_UID}")

---
## Step 5: Verify Cohorts with Federated Analytics

Run federated metrics against each cohort dataset to confirm the filters worked correctly.
All checks run as aggregate queries — no individual rows are returned.

> If cohort UIDs are not defined (e.g., after a kernel restart), paste the UIDs from the summary printed in Step 4 before running this cell. UIDs can also be pulled directly from the FCP UI by clicking the three dots to the right of the relevant line item and selecting "Copy UID"

In [ ]:
cohorts = [
    ("Males 30-40",   COHORT_MALES_UID),
    ("Females 30-40", COHORT_FEMALES_UID),
    ("Race 8515",     COHORT_RACE_UID),
]

for label, uid in cohorts:
    print(f"=== {label} ===")

    count_result = session.dataset.get_dataset_metric(uid, Count(variable="person_id"))
    n = count_result.output["count"]
    print(f"  Patients: {n:,}")

    if n > 0:
        age_result = session.dataset.get_dataset_metric(uid, Mean(variable="age"))
        print(f"  Mean age: {age_result.output['mean']:.1f}")

    print()

---
## Step 6: FCP UI — What to Check After Running This Notebook

1. **Code Objects** → Dashboard → Projects → [Your Project] → **Code**
   - Three Code Objects appear:
     - `Cohort Selection — Males 30-40`
     - `Cohort Selection — Females 30-40`
     - `Cohort Selection — Race 8515`
   - These are reusable — run them at additional sites with a single API call

2. **Code Runs** → Dashboard → Projects → [Your Project] → **Code Runs**
   - Three completed runs appear, one per cohort
   - Click any run → **Logs** to see input/output row counts and any warnings
   - If a cohort has 0 rows, the log will print a WARNING explaining why

3. **Output Datasets** → Dashboard → Projects → [Your Project] → **Datasets**
   - Three new datasets appear:
     - `Cohort — Males 30-40`
     - `Cohort — Females 30-40`
     - `Cohort — Race 8515`
   - Click any cohort → **Analytics** tab to inspect distributions:
     - Age-filtered cohorts: `age` column should show values only in [30, 40]
     - Males cohort: `is_male` should be 1 for all rows
     - Females cohort: `is_female` should be 1 for all rows
     - Race cohort: `race_concept_id` should be 8515 for all rows

---
## Summary — Copy These UIDs

In [ ]:
print("=" * 65)
print("  Tutorial 7 Complete — save these UIDs")
print("=" * 65)
print(f"COHORT_MALES_UID          = '{COHORT_MALES_UID}'")
print(f"COHORT_MALES_SCHEMA_UID   = '{COHORT_MALES_SCHEMA_UID}'")
print(f"COHORT_FEMALES_UID        = '{COHORT_FEMALES_UID}'")
print(f"COHORT_FEMALES_SCHEMA_UID = '{COHORT_FEMALES_SCHEMA_UID}'")
print(f"COHORT_RACE_UID           = '{COHORT_RACE_UID}'")
print(f"COHORT_RACE_SCHEMA_UID    = '{COHORT_RACE_SCHEMA_UID}'")
print("=" * 65)
print("\nContinue to: Tutorial 8 - Rhino MCP")